# Testing efficiency of number-specific optimizations

In [25]:
import numpy as np
import sys
sys.path.insert(0, '..') 
import src.cluster_number_operators as cluster_nums
import ffsim, tempfile
from src.dmrg_solver import Block2DMRGSolver
from pyblock2.driver import core
from pyblock2.driver.core import NPDMAlgorithmTypes
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

In [ ]:
memory_GB = 1 # need memory for computations of 2rdm
norb = 60
num_clusters = 5 # the exact clusters will be built randomly
nelec_tot = 60  # Na + Nb
bond_dim = 10
type_cost_function = 'variance'
maxiter = 10000
actual_rdms = False # if False, mps and RDMs computations are skipped; D, Gamma are initialized randomly. Allows to check performance of optimizer independently of RDM computation bottleneck. Notice that in that case the optimization landscape is wild, and you need more iterations to reach a local minimum

In [27]:

def random_cluster_matrix(k: int, norb: int) -> np.ndarray:
    """
    Generation of a k x norb cluster matrix.
    """
    # 1. Randomly assign each orbital to a cluster (0 to k-1) OR the ghost cluster (k)
    assignments = np.random.randint(0, k + 1, size=norb)
    
    # 2. Force at least one orbital into the ghost cluster to guarantee a zero-column
    assignments[np.random.randint(norb)] = k
    
    # 3. Allocate a (k+1) x norb matrix of zeros
    matrix = np.zeros((k + 1, norb), dtype=int)
    
    # 4. Vectorized one-hot assignment (places exactly one '1' in each column)
    matrix[assignments, np.arange(norb)] = 1
    
    # 5. Return only the first k rows. 
    # Columns assigned to 'k' will naturally be all zeros in this slice.
    return matrix[:k, :]

cluster_matrix = random_cluster_matrix(num_clusters, norb)
print(cluster_matrix)

[[0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
  0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 1
  0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0
  0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0
  1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0
  0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 1 1 0 0 0 0 0 1]]


In [28]:
if actual_rdms:
    tmp_dir = tempfile.mkdtemp(prefix='block2_test_')
    driver = DMRGDriver(scratch=tmp_dir, symm_type=SymmetryTypes.SZ, n_threads=1, stack_mem=int(memory_GB * 1024 * 1024 * 1024))
    driver.initialize_system(n_sites=norb, n_elec=nelec_tot, spin=0)
    mps = driver.get_random_mps(tag='RAND', bond_dim=bond_dim, nroots=1)
    print("got mps")

In [29]:
if actual_rdms:
    rdm1_a, rdm1_b = driver.get_1pdm(mps)
    print("got 1rdm")

In [30]:
if actual_rdms:
    rdm2_aa, rdm2_ab, rdm2_bb = driver.get_2pdm(mps,
                                                   site_type=0,
                                                   algo_type=NPDMAlgorithmTypes.SymbolFree | NPDMAlgorithmTypes.LowMem)
    print("got 2rdm")

In [31]:
if actual_rdms:
    D = rdm1_a + rdm1_b
    Gamma = rdm2_aa + rdm2_bb + rdm2_ab + rdm2_ab.transpose(1, 0, 3, 2)
else:
    D = np.random.rand(norb, norb)
    Gamma = np.random.rand(norb, norb, norb, norb)

In [32]:
from math import comb
import scipy.optimize
from src.orbital_rotation import params_to_U
import jax
import jax.numpy as jnp

if type_cost_function == 'variance':
    f = cluster_nums.number_variance_cost(D, Gamma, cluster_matrix)

if type_cost_function == 'eval_eq':
    # precompute the guessed eigenvalues
    clusters = cluster_nums.get_cluster_indices(cluster_matrix, norb, with_ghost=False) # number_eval_eq_cost attaches the ghost cluster number eigenvalue already; you do not need to precompute it
    evals = []
    for cluster in clusters:
        cluster_num_average = D[cluster, cluster].sum()
        evals.append(round(cluster_num_average))
    f = cluster_nums.number_eval_eq_cost(D, Gamma, cluster_matrix, evals)

# get with JAX function and gradient
f_val_and_grad = jax.jit(jax.value_and_grad(f))

# Wrap for SciPy (SciPy expects standard NumPy arrays and float64 returns)
def scipy_f(x):
    val, grad = f_val_and_grad(x)
    return float(val), jnp.asarray(grad, dtype=jnp.float64)

# Same, but no gradient
def scipy_f_nograd(x):
    val, grad = f_val_and_grad(x)
    return float(val)

# --- Step 3.2: Initial guess (identity rotation) ---
x0 = np.zeros(comb(norb, 2))
initial_cost = f(x0)
print(f"\nInitial cost (identity rotation): {initial_cost:.6e}")

# --- Step 3.3: Run optimization ---
# track iterations
class IterationTracker:
    def __init__(self):
        self.iteration = 0

    def __call__(self, intermediate_result):
        self.iteration += 1
        # Extract cost (fun) and current parameters (x)
        cost = intermediate_result.fun
        print(f"Iteration {self.iteration:3d} | Cost: {cost:.6f}")

tracker = IterationTracker()

print("\nRunning orbital optimization...")
result = scipy.optimize.minimize(
    fun=scipy_f,
    x0=x0,
    method='L-BFGS-B',
    jac=True,  # Tells SciPy that objective function returns (value, gradient)
    callback=tracker,
    options={'maxiter': maxiter, 'disp': True}
)
optimized_cost = result.fun
print(f"Optimized cost: {optimized_cost:.6e}")
print(f"Cost change: {(- initial_cost + optimized_cost)/initial_cost * 100:.2f}%")

# --- Step 3.4: Extract optimized orbital rotation ---
x_opt = result.x
U_opt = params_to_U(x_opt, norb)

# --- Step 3.5: Compare with randomly rotated orbitals ---
num_random_Us = 5
xs_random = [10 * np.random.rand(comb(norb, 2)) for _ in range(num_random_Us)]
random_costs = [f(x_random) for x_random in xs_random]
random_costs_perc = [int((- initial_cost + random_cost)/initial_cost * 100) for random_cost in random_costs]
print(f"\nFor reference, % cost change for some random unitaries: {', '.join(map(str, random_costs_perc))}%")


Initial cost (identity rotation): 1.856098e+02

Running orbital optimization...


/var/folders/r2/2ny_rjqj5w93mz1ll0gd2vbr0000gp/T/ipykernel_11341/162301905.py:52: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = scipy.optimize.minimize(


Iteration   1 | Cost: 180.488741
Iteration   2 | Cost: 40.312068
Iteration   3 | Cost: -329.938799
Iteration   4 | Cost: -468.105522
Iteration   5 | Cost: -690.257862
Iteration   6 | Cost: -974.674440
Iteration   7 | Cost: -1356.739719
Iteration   8 | Cost: -1546.941945
Iteration   9 | Cost: -1746.111401
Iteration  10 | Cost: -1887.398008
Iteration  11 | Cost: -2002.301149
Iteration  12 | Cost: -2071.031714
Iteration  13 | Cost: -2140.651782
Iteration  14 | Cost: -2202.186860
Iteration  15 | Cost: -2252.746128
Iteration  16 | Cost: -2303.813674
Iteration  17 | Cost: -2342.062417
Iteration  18 | Cost: -2373.181759
Iteration  19 | Cost: -2399.776407
Iteration  20 | Cost: -2420.947841
Iteration  21 | Cost: -2438.843533
Iteration  22 | Cost: -2454.668098
Iteration  23 | Cost: -2474.889732
Iteration  24 | Cost: -2488.087910
Iteration  25 | Cost: -2502.117745
Iteration  26 | Cost: -2515.199700
Iteration  27 | Cost: -2525.610202
Iteration  28 | Cost: -2535.461962
Iteration  29 | Cost: -2544.6

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# SECTION 4: Get labels of dominant sector and sectors obtained from it by 
# moving up to num_transfers electrons
# ============================================================================

In [ ]:
# ============================================================================
# SECTION 5: For small systems, evaluate quality of sector decomposition
# by plotting K_sectors -> ΔE, K_states -> ΔE
# ============================================================================